# Build Your Own AI Agent

## Workshop Part 3: Hands-On Agent Building with LangGraph

Learn to build AI agents using a fluent API that wraps LangGraph complexity.

### Learning Objectives
By the end of this session, you will be able to:
1. Add pre-built tools to an agent
2. Create custom tools with routing rules
3. Enable conversation memory
4. Build a complete multi-tool agent

### Time Estimate: 50 minutes hands-on

## 1. Setup and Dependencies

In [ ]:
%pip install langchain-core langgraph -q

In [ ]:
# Add src to path for imports
import os
import sys

# For Databricks notebooks
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    # Get the workspace path
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    # For local development
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print(f"Python path configured. Project root: {sys.path[0]}")

## 2. Understanding the AgentBuilder Architecture

The `AgentBuilder` provides a fluent API for constructing LangGraph agents:

```
+---------------------+
|    AgentBuilder     |    Fluent API for configuration
+---------------------+
         |
         | .set_system_prompt()
         | .add_tool()
         | .add_routing_rule()
         | .enable_memory()
         | .build()
         v
+---------------------+
|  LangGraph          |    StateGraph with:
|  StateGraph         |    - Agent node (decision making)
|                     |    - Tool node (execution)
+---------------------+    - Routing edges
         |
         v
+---------------------+
|   AgentRunner       |    High-level query interface
|   .query()          |    Handles state management
|   .get_history()    |    Supports memory/persistence
+---------------------+
```

**Mock Mode vs Real Mode:**
- **Mock Mode**: Uses keyword-based routing (no LLM calls needed)
- **Real Mode**: Uses Databricks LLM for intelligent tool selection

In [ ]:
from src.workshop import AgentBuilder, calculator, date_helper, mock_web_search

# Quick demo of available pre-built tools
print("Available pre-built tools:")
print("=" * 50)
print("- calculator: Math expressions (2+2, sqrt(16), 15% of 250)")
print("- date_helper: Date/time queries (today, days until Christmas)")
print("- mock_web_search: Simulated web search for testing")
print()
print("AgentBuilder methods:")
print("=" * 50)
print(".set_system_prompt(prompt)  - Define agent personality")
print(".add_tool(name, desc, func) - Add a tool to the agent")
print(".add_routing_rule(kw, tool) - Route queries by keywords")
print(".enable_memory(thread_id)   - Enable conversation memory")
print(".build(mock_mode=True)      - Build and return the agent")

## 3. Exercise 1: Add a Calculator Tool (10 min)

**Goal:** Build your first agent with a single tool.

The `calculator` tool can:
- Evaluate expressions: `2 + 2`, `10 * 5`
- Handle percentages: `15% of 250`
- Use functions: `sqrt(16)`, `round(3.7)`

**Your Task:**
1. Uncomment the `add_tool()` call
2. Fill in the `name` and `description`
3. Run the cell and verify the output

In [ ]:
# ===== EXERCISE 1: Add a Calculator Tool =====
# Time: 10 minutes
#
# Goal: Add the pre-built calculator tool to an agent
#
# Instructions:
# 1. Uncomment the add_tool() call below
# 2. Fill in the name and description
# 3. Run this cell and test with the query


builder = AgentBuilder("Calculator Agent")
builder.set_system_prompt("You are a helpful math assistant.")

# TODO: Uncomment and complete this:
# builder.add_tool(
#     name="???",           # Hint: use "calculator"
#     description="???",    # Hint: describe what the tool does
#     function=calculator.func,
# )

# Add routing rule for mock mode
builder.add_routing_rule(
    keywords=["calculate", "what is", "+", "-", "*", "/", "percent"],
    tool_name="calculator",
)

agent = builder.build(mock_mode=True)

# Test your agent
response = agent.query("What is 15% of 250?")
print(f"Response: {response}")
# Expected: 37.5

## 4. Exercise 2: Multi-Tool Agent with Routing (15 min)

**Goal:** Build an agent with multiple tools and priority-based routing.

Routing rules determine which tool handles a query:
- Higher `priority` values are checked first
- Keywords are matched case-insensitively
- First matching rule wins

**Your Tasks:**
1. Add both `calculator` and `date_helper` tools
2. Set appropriate routing priorities
3. Test with different query types

In [ ]:
# ===== EXERCISE 2: Multi-Tool Agent with Routing =====
# Time: 15 minutes
#
# Goal: Build an agent with multiple tools and priority-based routing
#
# Instructions:
# 1. Add the date_helper tool (already started for you)
# 2. Set routing priorities so date queries route correctly
# 3. Test with the queries below

from src.workshop import AgentBuilder

builder = AgentBuilder("Multi-Tool Assistant")
builder.set_system_prompt("You are a helpful assistant that can do math and answer date questions.")

# Tool 1: Calculator (provided)
builder.add_tool(
    name="calculator",
    description="Evaluate mathematical expressions like 2+2, sqrt(16), or 15% of 250",
    function=calculator.func,
)

# TODO: Add Tool 2 - date_helper
# builder.add_tool(
#     name="???",
#     description="???",
#     function=date_helper.func,
# )

# Routing rules - which keywords trigger which tool?
# TODO: Set appropriate priorities (higher = checked first)

builder.add_routing_rule(
    keywords=["calculate", "math", "+", "-", "*", "/", "percent", "sqrt"],
    tool_name="calculator",
    priority=0,  # TODO: Adjust if needed
)

# TODO: Add routing rule for date_helper
# builder.add_routing_rule(
#     keywords=["???", "???", "???"],
#     tool_name="date_helper",
#     priority=???,
# )

agent = builder.build(mock_mode=True)

# Test queries
print("Testing Multi-Tool Agent")
print("=" * 50)

test_queries = [
    "What is 42 * 17?",
    "What day is today?",
    "How many days until Christmas?",
    "Calculate sqrt(144)",
]

for query in test_queries:
    response = agent.query(query, reset_history=True)
    print(f"Q: {query}")
    print(f"A: {response}")
    print()

## 5. Exercise 3: Conversation Memory (10 min)

**Goal:** Enable conversation memory so the agent remembers context.

Without memory, each query is independent. With memory:
- The agent maintains conversation history
- Follow-up questions work naturally
- Context is preserved across queries

**Your Task:**
1. Enable memory with `enable_memory()`
2. Run the conversation sequence
3. Observe how context is maintained

In [ ]:
# ===== EXERCISE 3: Conversation Memory =====
# Time: 10 minutes
#
# Goal: Enable conversation memory so the agent remembers context
#
# Instructions:
# 1. Uncomment the enable_memory() call
# 2. Run the conversation and observe context retention
# 3. Try adding your own follow-up questions

from src.workshop import AgentBuilder, calculator

builder = AgentBuilder("Memory-Enabled Assistant")
builder.set_system_prompt("You are a helpful assistant with conversation memory.")

# Add tools
builder.add_tool("calculator", "Evaluate math expressions", calculator.func)
builder.add_tool("date_helper", "Answer date/time questions", date_helper.func)

# Add routing rules
builder.add_routing_rule(["calculate", "math", "+", "-", "*", "/"], "calculator")
builder.add_routing_rule(["today", "date", "time", "tomorrow", "yesterday"], "date_helper")

# TODO: Enable memory with a unique thread ID
# builder.enable_memory(thread_id="my-session-1")

agent = builder.build(mock_mode=True)

# Conversation sequence
print("Testing Conversation Memory")
print("=" * 50)

# First query
response1 = agent.query("What is 100 + 50?")
print("Q: What is 100 + 50?")
print(f"A: {response1}")
print()

# Second query (without reset)
response2 = agent.query("What day is today?")
print("Q: What day is today?")
print(f"A: {response2}")
print()

# Check conversation history
history = agent.get_history()
print(f"Conversation history has {len(history)} messages")
print()

# Clear history and try again
agent.clear_history()
print("History cleared!")
print(f"Conversation history now has {len(agent.get_history())} messages")

## 6. Exercise 4: Playground - Build Your Own Agent (15 min)

**Goal:** Experiment with custom configurations. Build a custom agent with your own settings.

**Ideas to try:**
- Add all three pre-built tools
- Experiment with different routing priorities
- Create complex keyword patterns
- Test edge cases

**Challenge:** Can you create an agent that handles:
1. "What is 20% of the number of days until Christmas?"
2. "Search for today's weather forecast"

In [ ]:
# ===== EXERCISE 4: Playground - Build Your Own Agent =====
# Time: 15 minutes
#
# Goal: Experiment! Build any agent configuration you want.
#
# Try these challenges:
# 1. Add all three tools (calculator, date_helper, mock_web_search)
# 2. Create routing rules with different priorities
# 3. Test queries that could match multiple tools

from src.workshop import AgentBuilder, calculator, date_helper

# Your custom agent
builder = AgentBuilder("My Custom Agent")

# TODO: Configure your agent!
# builder.set_system_prompt("...")
# builder.add_tool(...)
# builder.add_routing_rule(...)
# builder.enable_memory(...)

# Example: Complete agent with all tools
# builder.set_system_prompt("You are a versatile assistant.")
# builder.add_tool("calculator", "Evaluate math", calculator.func)
# builder.add_tool("date_helper", "Date queries", date_helper.func)
# builder.add_tool("web_search", "Search the web", mock_web_search.func)
# builder.add_routing_rule(["calculate", "+", "-"], "calculator", priority=1)
# builder.add_routing_rule(["today", "date", "christmas"], "date_helper", priority=2)
# builder.add_routing_rule(["search", "find", "weather"], "web_search", priority=0)
# builder.enable_memory("playground-session")

# agent = builder.build(mock_mode=True)

# Test your agent
# response = agent.query("Your test query here")
# print(response)

In [ ]:
# Pre-built example: Full-featured agent to play with
from src.workshop import AgentBuilder, calculator, date_helper

full_agent = (
    AgentBuilder("Full-Featured Assistant")
    .set_system_prompt("You are a versatile assistant that can do math, answer date questions, and search the web.")
    .add_tool("calculator", "Evaluate mathematical expressions", calculator.func)
    .add_tool("date_helper", "Answer date and time questions", date_helper.func)
    .add_tool("web_search", "Search the web for information", mock_web_search.func)
    .add_routing_rule(
        ["calculate", "math", "+", "-", "*", "/", "percent", "sqrt"],
        "calculator",
        priority=1,
    )
    .add_routing_rule(
        ["today", "date", "time", "tomorrow", "yesterday", "christmas", "new year"],
        "date_helper",
        priority=2,
    )
    .add_routing_rule(
        ["search", "find", "look up", "weather", "news"],
        "web_search",
        priority=0,
    )
    .enable_memory("demo-session")
    .build(mock_mode=True)
)

# Interactive queries - modify and run!
queries = [
    "What is 25% of 400?",
    "How many days until New Year?",
    "Search for the latest tech news",
    "What's the weather forecast?",
]

print("Full-Featured Agent Demo")
print("=" * 50)

for q in queries:
    print(f"\nQ: {q}")
    response = full_agent.query(q, reset_history=True)
    # Show first 200 chars for brevity
    preview = response[:200] + "..." if len(response) > 200 else response
    print(f"A: {preview}")

## 7. Architecture Summary

### What You Built

```
                Your Agent
                    |
     +--------------+---------------+
     |              |               |
     v              v               v
+--------+   +------------+   +-----------+
|  calc  |   | date_helper|   |web_search |
+--------+   +------------+   +-----------+
  math         dates           search
  queries      queries         queries
```

### LangGraph Under the Hood

```
                     START
                       |
                       v
              +----------------+
              |  Agent Node    |<---------+
              | (LLM/Router)   |          |
              +----------------+          |
                   |      |               |
          use tool |      | respond       |
                   v      v               |
            +--------+  END               |
            | Tools  |                    |
            |  Node  |--------------------+
            +--------+    (loop back)
```

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Tool** | A function the agent can call |
| **Routing Rule** | Keyword-based tool selection (mock mode) |
| **Memory** | Conversation history persistence |
| **StateGraph** | LangGraph's core abstraction for agent workflows |

## Bonus Challenge: Create a Custom Tool

Additional practice: Try creating your own tool.

```python
from langchain_core.tools import tool

@tool
def my_tool(query: str) -> str:
    """Description of what your tool does."""
    # Your logic here
    return f"Processed: {query}"

# Then add it to your agent:
# builder.add_tool("my_tool", "My tool description", my_tool.func)
```

**Ideas for custom tools:**
- Unit converter (km to miles, Celsius to Fahrenheit)
- Simple dictionary lookup
- String manipulation (reverse, uppercase, etc.)

In [ ]:
# View exercise solutions (run if you get stuck)
# from src.workshop.solutions import test_all_solutions
# test_all_solutions()

# Or view just the solution code:
# from src.workshop.solutions import EXERCISE_SOLUTIONS
# print(EXERCISE_SOLUTIONS['exercise_1'])

## Next Steps

### Continue Learning

1. **Basic Demo**: See a complete multi-agent system in action
   - [demo.ipynb](./demo.ipynb)

2. **Advanced Demo**: Multi-Genie orchestration with parallel queries
   - [advanced_demo.ipynb](./advanced_demo.ipynb)

### Additional Resources

- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [LangChain Tools Guide](https://python.langchain.com/docs/modules/tools/)
- [Databricks LangChain Integration](https://docs.databricks.com/en/machine-learning/llm/langchain.html)

### Workshop Solutions

```python
# View exercise solutions
from src.workshop.solutions import test_all_solutions
test_all_solutions()
```

In [ ]:
# Cleanup - reset any state
print("Workshop complete!")
print()
print("Key takeaways:")
print("1. AgentBuilder provides a fluent API for LangGraph agents")
print("2. Tools are functions your agent can call")
print("3. Routing rules enable keyword-based tool selection")
print("4. Memory preserves conversation context")
print()
print("See demo.ipynb for a complete multi-agent system.")